In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Ranges are INCLUSIVE [start, end]
# We map the *Filename Stem* (or key part of it) to the config
SPLIT_CONFIG = {
    CommSignal2: {
        "train": (0, 69),    # 70 files
        "val":   (70, 84),   # 15 files
        "test":  (85, 99)    # 15 files
    },
    EMISignal1: {
        "train": (0, 370),
        "val":   (371, 450),
        "test":  (451, 529)
    },
    CommSignal3: {
        "train": (0, 97),
        "val":   (98, 118),
        "test":  (119, 138)
    },
    CommSignal5G1: {
        "train": (0, 104),
        "val":   (105, 126),
        "test":  (127, 148)
    },
    BarrageSignal: {
        "train": (0, 699),
        "val":   (700, 849),
        "test":  (850, 999)
    } 
}

In [ ]:
SPLIT_DIRS = {
    "train": UNET_TRAIN_DIR,
    "val":   UNET_VAL_DIR,
    "test":  UNET_TEST_DIR
}

for d in SPLIT_DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
def complex_to_iq_tensor(data):
    """
    Converts (N, L) complex array to (N, 2, L) float array.
    Channel 0 = Real (I), Channel 1 = Imag (Q).
    """
    # extract real and imag parts
    real_part = data.real.astype(np.float32)
    imag_part = data.imag.astype(np.float32)
    
    # Stack along axis 1 to go from (N, L) -> (N, 2, L)
    return np.stack([real_part, imag_part], axis=1)

In [ ]:
manifest_records = []

print(f"Starting Splitting: {len(UNET_DATASET_CLEAN)} source files...\n")

for file_path in UNET_DATASET_CLEAN:
    filename = file_path.name
    
    # Simple check to match config keys (e.g. "CommSignal2" in "CommSignal2_raw_data_clean.npy")
    signal_key = next((key for key in SPLIT_CONFIG.keys() if key in filename), None)
    
    if not signal_key:
        print(f"Skipping {filename}: Could not match to a configuration key.")
        continue
        
    print(f"Processing: {signal_key} ({filename})")
    
    if not file_path.exists():
        print(f"Error: File not found at {file_path}")
        continue
        
    full_data = np.load(file_path)
    total_samples = full_data.shape[0]
    print(f"Loaded shape: {full_data.shape}")
    
    # Barrage is already in (N, 2, L) format, so we check for that and skip transformation if needed
    is_pre_formatted = (full_data.ndim == 3 and full_data.shape[1] == 2)

    # Get the specific split rules for this signal
    rules = SPLIT_CONFIG[signal_key]
    
    for split_name, (start_idx, end_idx) in rules.items():
        if end_idx >= total_samples:
            print(f"Warning: Config end index {end_idx} exceeds file length {total_samples}. Clipping.")
            end_idx = total_samples - 1
            
        split_data = full_data[start_idx : end_idx + 1]

        if is_pre_formatted:
            split_data_transformed = split_data
        else:
            split_data_transformed = complex_to_iq_tensor(split_data)

        clean_name = f"{signal_key}_{split_name}.npy"
        out_path = SPLIT_DIRS[split_name] / clean_name
        
        np.save(out_path, split_data_transformed)
        
        print(f"{split_name.upper()}: Saved {clean_name} (Rows {start_idx}-{end_idx}, Count: {split_data.shape[0]})")
        print(f"saved as shape:{split_data_transformed.shape}")
    
    print("-" * 40)